# Top 3 Model Performance By Train-Test Similarity Bin

This notebook asks whether the top scaffold-CV models perform differently on test molecules that are chemically far from, moderately similar to, or very close to their nearest training-set molecule.

The model is **not retrained per bin**. Each model is trained normally on its full training fold. After prediction, test molecules are grouped by nearest-neighbor Morgan Tanimoto similarity to the training set, and metrics are calculated separately within each similarity bin.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

ROOT = Path.cwd()
if ROOT.name != "brainroute_ml_validation":
    ROOT = ROOT / "brainroute_ml_validation"

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")

## Load Model Metrics, Predictions, And Similarity Bins

In [ ]:
perf = pd.read_csv(REPORTS / "model_performance_all_splits.csv")
preds = pd.read_csv(REPORTS / "model_predictions_all_splits.csv")
near = pd.read_csv(
    REPORTS / "near_duplicate_analysis_all_splits.csv",
    usecols=["molecule_id", "split", "max_tanimoto", "similarity_bin"],
)

print(perf.shape, preds.shape, near.shape)
perf.head()

## Select The Top 3 Models By Mean Scaffold-CV Balanced Accuracy

The top models are selected using scaffold 5-fold CV, which is the primary validation protocol.

In [ ]:
primary = perf[perf["split"].str.startswith("scaffold_cv_fold", na=False)].copy()

ranking = (
    primary.groupby(["feature_view", "model"])
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        auprc_mean=("auprc", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        n=("balanced_accuracy", "size"),
    )
    .reset_index()
    .sort_values(["balanced_accuracy_mean", "auprc_mean"], ascending=False)
)

top3 = ranking.head(3).copy()
top3

## Merge Test Predictions With Similarity Bins

The merge key is `molecule_id + split`. Each row remains a test-set prediction from a trained model, now annotated with the test molecule's nearest-training Tanimoto similarity and bin.

In [ ]:
pred_scaffold = preds[preds["split"].str.startswith("scaffold_cv_fold", na=False)].copy()
merged = pred_scaffold.merge(near, on=["molecule_id", "split"], how="left")
merged = merged.merge(top3[["feature_view", "model"]], on=["feature_view", "model"], how="inner")

merged[["feature_view", "model", "split", "molecule_id", "y_true", "y_pred", "y_score", "max_tanimoto", "similarity_bin"]].head()

## Compute Bin-Wise Metrics

Balanced accuracy is calculated separately within each similarity bin. This shows whether model performance is concentrated among close analogs.

In [ ]:
def metric_row(group):
    y_true = group["y_true"].astype(int).to_numpy()
    y_pred = group["y_pred"].astype(int).to_numpy()
    y_score = group["y_score"].to_numpy()
    both_classes = len(set(y_true)) > 1
    return pd.Series(
        {
            "n": len(group),
            "n_bbb_minus": int((y_true == 0).sum()),
            "n_bbb_plus": int((y_true == 1).sum()),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred) if both_classes else np.nan,
            "accuracy": accuracy_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "mcc": matthews_corrcoef(y_true, y_pred) if both_classes else np.nan,
            "roc_auc": roc_auc_score(y_true, y_score) if both_classes else np.nan,
            "auprc": average_precision_score(y_true, y_score) if both_classes else np.nan,
            "mean_max_tanimoto": group["max_tanimoto"].mean(),
            "median_max_tanimoto": group["max_tanimoto"].median(),
        }
    )

bin_order = {"<0.40": 0, "0.40_to_0.60": 1, "0.60_to_0.80": 2, ">0.80": 3}

bin_summary = (
    merged.groupby(["feature_view", "model", "similarity_bin"], observed=True)
    .apply(metric_row, include_groups=False)
    .reset_index()
)
bin_summary["bin_order"] = bin_summary["similarity_bin"].map(bin_order)
bin_summary = bin_summary.sort_values(["feature_view", "model", "bin_order"]).drop(columns=["bin_order"])

out_path = REPORTS / "top3_scaffold_cv_performance_by_similarity_bin.csv"
bin_summary.to_csv(out_path, index=False)
print(out_path)
bin_summary

## Plot Balanced Accuracy By Similarity Bin

In [ ]:
plot_df = bin_summary.copy()
plot_df["model_label"] = plot_df["feature_view"] + " / " + plot_df["model"]
plot_df["similarity_bin"] = pd.Categorical(
    plot_df["similarity_bin"],
    categories=["<0.40", "0.40_to_0.60", "0.60_to_0.80", ">0.80"],
    ordered=True,
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=plot_df,
    x="similarity_bin",
    y="balanced_accuracy",
    hue="model_label",
    marker="o",
    linewidth=2,
    ax=ax,
)
ax.set_xlabel("Nearest-training Morgan Tanimoto similarity bin")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Top 3 Scaffold-CV Models: Performance By Similarity Bin")
ax.legend(title="Model", fontsize=8, title_fontsize=9, frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(FIGURES / "top3_scaffold_cv_balanced_accuracy_by_similarity_bin.png", dpi=240, bbox_inches="tight")
plt.show()